# Phase 6c — cuBLAS GEMM speedup (verify correctness, then time COCO128)
Adds a cuBLAS path to bk::gemm (build `-DUSE_CUBLAS -lcublas`). **Runtime → GPU**, Run all.


In [ ]:
!nvidia-smi -L
!nvcc --version | tail -1


In [ ]:
%cd /content
!rm -rf yolov8_cpp
!git clone -q --branch main https://github.com/yomei-o/yolov8_cpp.git
%cd /content/yolov8_cpp


### 1. Correctness: full forward with cuBLAS must still match the CPU engine (~3e-5)


In [ ]:
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -DUSE_CUBLAS -Ipure/third_party pure/dnet_test.cpp -lcublas -o dnet_cublas
!./dnet_cublas 2>/dev/null; echo "exit=$?"


### 2. Get COCO128


In [ ]:
!wget -q https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip && unzip -q -o coco128.zip


### 3. Speed: COCO128 training with cuBLAS (compare s/epoch to naive-GPU 113s, hosted 276s)


In [ ]:
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -DUSE_CUBLAS -Ipure/third_party pure/dtrain_coco.cpp -lcublas -o dtrain_coco_cublas
!./dtrain_coco_cublas coco128/images/train2017 640 4 3


---
**Look for**: (1) forward `worst |device-CPU| ~3e-5 MATCH` (confirms the cuBLAS gemm mapping is correct);
(2) the new `s/epoch` vs naive-GPU **113 s/ep** and hosted **276 s/ep**. Loss should still decrease.
